# IS 455 — pipeline notebook (template)

## TA run instructions
- Set **`PIPELINE_NAME`** in the next code cell to one of the registered pipelines (see list printed when you run).
- **Working directory** should be `ml-pipelines/` so `../datasets/` resolves correctly.
- **Kernel → Restart & Run All**.

Shared modeling logic lives in **`pipeline_kit.py`** (explanatory linear/logistic + predictive random forest, metrics, coefficient/importance tables, optional CSV exports).

### All Tier A pipelines (`PIPELINE_NAME` → primary notebook)
1. `residents` → `residents.ipynb` · 2. `safehouses` → `safehouses.ipynb` · 3. `donation_allocations` → `donation_allocations.ipynb` · 4. `intervention_plans` → `intervention_plans.ipynb` · 5. `incident_reports` → `incident_reports.ipynb` · 6. `partner_assignments` → `partner_assignments.ipynb` · 7. `education_records` → `education_records.ipynb` · 8. `health_wellbeing_records` → `health_wellbeing_records.ipynb` · 9. `social_media_posts` → `social_media_posts_pipeline.ipynb` · 10. `process_recordings` → `process_recordings.ipynb` · 11. `home_visitations` → `home_visitations.ipynb`

Full checklist: **`ALL_PIPELINES_ROADMAP.md`**. Batch refresh: **`run_all_pipelines.ipynb`**.

## 1. Problem framing (Ch. 1)

Edit this section for your submission narrative.

- **Business question:**
- **Stakeholder:**
- **Predictive goal:**
- **Explanatory goal:**
- **Success metrics:**
- **Why it matters:**

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Ensure imports work when cwd is ml-pipelines/
NOTEBOOK_DIR = Path.cwd().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline_kit import NotebookConfig, PIPELINE_REGISTRY, run_notebook_driver, datasets_dir

# --- Choose pipeline (must match a key in PIPELINE_REGISTRY) ---
PIPELINE_NAME = "residents"  # e.g. "donation_allocations", "education_records", ...

print("Registered pipelines:", sorted(PIPELINE_REGISTRY.keys()))

cfg = NotebookConfig(pipeline_name=PIPELINE_NAME, seed=42, save_artifacts=True)
result = run_notebook_driver(cfg)

print("\nTask:", result["task"])
print("Meta:", result["meta"])

Registered pipelines: ['donation_allocations', 'education_records', 'health_wellbeing_records', 'home_visitations', 'incident_reports', 'intervention_plans', 'partner_assignments', 'process_recordings', 'residents', 'safehouses', 'social_media_posts']
=== residents ===
current_risk_level (multiclass)
Shape X=(60, 44), task=classification, n=60

-- Metrics --
  explanatory_accuracy: 0.6000
  explanatory_balanced_accuracy: 0.4000
  explanatory_weighted_f1: 0.6000
  predictive_accuracy: 0.6000
  predictive_balanced_accuracy: 0.3000
  predictive_weighted_f1: 0.5000

Top linear drivers (explanatory, |coef| proxy):


,feature,abs_coefficient
0,num__sub_cat_child_labor,0.301147
1,cat__initial_risk_level_Low,0.292892
2,num__family_is_4ps,0.246259
3,cat__initial_risk_level_Medium,0.224269
4,cat__initial_risk_level_High,0.204758
5,cat__initial_case_assessment_For Foster Care,0.204703
6,num__sub_cat_at_risk,0.204213
7,cat__referral_source_Community,0.193640
8,cat__initial_case_assessment_For Continued Care,0.187371
9,num__has_special_needs,0.184779



Top random forest importances (predictive):


,feature,importance
0,cat__initial_risk_level_Low,0.035306
1,num__length_of_stay_months,0.031990
2,num__age_upon_admission_months,0.031268
3,num__present_age_months,0.028577
4,num__safehouse_id,0.024902
5,num__family_is_4ps,0.024453
6,cat__initial_risk_level_Medium,0.022723
7,num__days_enrolled_to_closed,0.020771
8,cat__referral_source_Community,0.018501
9,cat__initial_risk_level_High,0.016159



Saved artifacts under C:\Users\abiga\IntextW2026\ml-pipelines\artifacts

Task: classification
Meta: {'name': 'residents', 'target_description': 'current_risk_level (multiclass)'}


## 2. Data acquisition, preparation & exploration (Ch. 2–5, 7)

- **Sources:** CSV under `datasets/` (see `pipeline_kit.prepare_*` for the exact target and columns).
- **Preparation:** Implemented in `pipeline_kit.py` for reproducibility; extend or copy functions if you need custom joins.
- **Exploration:** Use the cell below to inspect raw rows, missingness, and simple plots.

In [2]:
# Quick peek at the source CSV for the selected pipeline
csv_map = {
    "residents": "residents.csv",
    "safehouses": "safehouses.csv",
    "donation_allocations": "donation_allocations.csv",
    "intervention_plans": "intervention_plans.csv",
    "incident_reports": "incident_reports.csv",
    "partner_assignments": "partner_assignments.csv",
    "education_records": "education_records.csv",
    "health_wellbeing_records": "health_wellbeing_records.csv",
    "social_media_posts": "social_media_posts.csv",
    "process_recordings": "process_recordings.csv",
    "home_visitations": "home_visitations.csv",
}
fname = csv_map.get(PIPELINE_NAME)
if fname:
    raw = pd.read_csv(datasets_dir() / fname)
    display(raw.head(8))
    display(raw.isna().mean().sort_values(ascending=False).head(20))
else:
    print("No CSV mapping for", PIPELINE_NAME)

,resident_id,case_control_no,internal_code,safehouse_id,case_status,sex,date_of_birth,birth_status,place_of_birth,religion,...,initial_case_assessment,date_case_study_prepared,reintegration_type,reintegration_status,initial_risk_level,current_risk_level,date_enrolled,date_closed,created_at,notes_restricted
0,1,C0043,LS-0001,4,Active,F,2008-08-31,Marital,Davao City,Unspecified,...,For Reunification,2023-12-14,Foster Care,In Progress,Critical,High,2023-10-17,NaN,2023-10-17 00:00:00,NaN
1,2,C2530,LS-0002,3,Closed,F,2008-04-23,Marital,Cebu City,Seventh-day Adventist,...,For Continued Care,2023-04-10,Family Reunification,Completed,Medium,Medium,2023-03-18,2025-01-06,2023-03-18 00:00:00,NaN
2,3,C3946,LS-0003,1,Active,F,2007-01-31,Marital,Manila,Roman Catholic,...,For Independent Living,NaN,Foster Care,Completed,Medium,Medium,2024-05-24,NaN,2024-05-24 00:00:00,NaN
3,4,C3116,LS-0004,2,Active,F,2012-06-29,Marital,Davao City,Evangelical,...,For Reunification,2024-10-25,NaN,On Hold,High,Low,2024-09-27,NaN,2024-09-27 00:00:00,NaN
4,5,C9132,LS-0005,4,Transferred,F,2009-04-20,Marital,Pasay City,Buddhism,...,For Independent Living,2024-02-14,Family Reunification,Completed,Medium,Low,2024-01-10,2024-10-08,2024-01-10 00:00:00,NaN
5,6,C7286,LS-0006,5,Active,F,2006-05-19,Non-Marital,Pasay City,Seventh-day Adventist,...,For Reunification,2023-09-29,Family Reunification,In Progress,High,Medium,2023-08-30,NaN,2023-08-30 00:00:00,NaN
6,7,C6898,LS-0007,6,Active,F,2015-09-17,Marital,Zamboanga City,Evangelical,...,For Reunification,NaN,Independent Living,In Progress,Low,Low,2024-06-22,NaN,2024-06-22 00:00:00,NaN
7,8,C3744,LS-0008,4,Active,F,2012-02-15,Marital,Iloilo City,Seventh-day Adventist,...,For Independent Living,2024-03-12,Adoption (Domestic),Not Started,High,High,2024-02-28,NaN,2024-02-28 00:00:00,NaN


notes_restricted            1.000000
pwd_type                    0.950000
special_needs_diagnosis     0.900000
date_closed                 0.500000
referring_agency_person     0.400000
date_colb_obtained          0.400000
date_colb_registered        0.216667
date_case_study_prepared    0.183333
reintegration_type          0.083333
safehouse_id                0.000000
family_parent_pwd           0.000000
family_informal_settler     0.000000
date_of_admission           0.000000
age_upon_admission          0.000000
present_age                 0.000000
length_of_stay              0.000000
referral_source             0.000000
assigned_social_worker      0.000000
case_status                 0.000000
initial_case_assessment     0.000000
dtype: float64

## 3. Modeling & feature selection (Ch. 9–14, 16)

- **Explanatory:** `LinearRegression` (regression) or `LogisticRegression` (classification) on preprocessed features.
- **Predictive:** `RandomForestRegressor` / `RandomForestClassifier`.
- **Selection:** Review `explanatory_coefs` and `predictive_importance` tables from the run; refine with domain drops (IDs, leakage) in `pipeline_kit.py` if needed.

In [3]:
# Driver tables (also printed during run_notebook_driver)
display(result["explanatory_coefs"])
display(result["predictive_importance"])

,feature,abs_coefficient
0,num__sub_cat_child_labor,0.301147
1,cat__initial_risk_level_Low,0.292892
2,num__family_is_4ps,0.246259
3,cat__initial_risk_level_Medium,0.224269
4,cat__initial_risk_level_High,0.204758
5,cat__initial_case_assessment_For Foster Care,0.204703
6,num__sub_cat_at_risk,0.204213
7,cat__referral_source_Community,0.193640
8,cat__initial_case_assessment_For Continued Care,0.187371
9,num__has_special_needs,0.184779


,feature,importance
0,cat__initial_risk_level_Low,0.035306
1,num__length_of_stay_months,0.031990
2,num__age_upon_admission_months,0.031268
3,num__present_age_months,0.028577
4,num__safehouse_id,0.024902
5,num__family_is_4ps,0.024453
6,cat__initial_risk_level_Medium,0.022723
7,num__days_enrolled_to_closed,0.020771
8,cat__referral_source_Community,0.018501
9,cat__initial_risk_level_High,0.016159


## 4. Evaluation & interpretation (Ch. 15)

- **Metrics:** Printed above (`metrics` dict). Regression: MAE, R². Classification: accuracy, balanced accuracy, weighted F1.
- **Write in your own words:** What would a false positive or false negative mean for the organization?
- **Limitations:** Small sample sizes (e.g. safehouses n=9) inflate variance; note in your report.

In [4]:
import json

print(json.dumps(result["metrics"], indent=2))
if "classification_report_predictive" in result:
    print(result["classification_report_predictive"])

{
  "explanatory_accuracy": 0.6,
  "explanatory_balanced_accuracy": 0.39999999999999997,
  "explanatory_weighted_f1": 0.6,
  "predictive_accuracy": 0.6,
  "predictive_balanced_accuracy": 0.3,
  "predictive_weighted_f1": 0.5
}
              precision    recall  f1-score   support

        High       0.00      0.00      0.00         1
         Low       0.64      0.90      0.75        10
      Medium       0.00      0.00      0.00         4

    accuracy                           0.60        15
   macro avg       0.21      0.30      0.25        15
weighted avg       0.43      0.60      0.50        15



## 5. Causal and relationship analysis (required narrative)

- **Do not** equate predictive accuracy with causation.
- Use the coefficient and importance tables to **hypothesize** drivers; discuss confounders and data limits.
- **Recommendations:** Operational, ethical, and realistic for the organization.

In [5]:
# Optional: export interpretation tables for your write-up (artifacts already saved when save_artifacts=True)
from pipeline_kit import artifact_dir

print("Artifacts:", artifact_dir())

Artifacts: C:\Users\abiga\IntextW2026\ml-pipelines\artifacts
